In [0]:
%sql
CREATE OR REPLACE VIEW sts_prd.`01_ipd_thering`.g_bowler_ww_orderfcst AS
WITH material_set AS (
  SELECT DISTINCT Product, Level1Desc
  FROM hub_prd.g_bpc_finance.v_reltioproduct_hier
  WHERE Level1Desc IN (
    'Alaris System','Hazardous Drug Solutions','IV Solutions',
    'Gravity and Syringe Delivery','IV Access','CME',
    'Alaris Plus','IPD OEM','BD Nexus'
  )
  AND Product IS NOT NULL
),
bounds AS (
  SELECT
    date_format(add_months(trunc(add_months(current_date, -9), 'YYYY'),  9), 'yyyyMM') AS fy_start_yyyymm,
    date_format(add_months(trunc(add_months(current_date, -9), 'YYYY'), 21), 'yyyyMM') AS fy_end_yyyymm
),
filtered AS (
  SELECT f.*
  FROM hub_prd.s_shared.scm_fcst f
  JOIN material_set m
    ON f.Product = m.Product
  CROSS JOIN bounds b
  WHERE CAST(f.CalMonth AS STRING) BETWEEN b.fy_start_yyyymm AND b.fy_end_yyyymm
),
snap_max AS (
  -- Compute the last SnapshotDate for each *snapshot month*
  SELECT
    trunc(SnapshotDate, 'MM') AS snapshot_month,
    MAX(SnapshotDate)         AS last_snapshot_in_month
  FROM filtered
  GROUP BY trunc(SnapshotDate, 'MM')
)
SELECT f.*
FROM filtered f
JOIN snap_max s
  ON trunc(f.SnapshotDate, 'MM') = s.snapshot_month
 AND f.SnapshotDate             = s.last_snapshot_in_month;

In [0]:
%sql
CREATE OR REPLACE VIEW sts_prd.`01_ipd_thering`.g_bowler_ww_orderdemand AS
WITH material_set AS (
  SELECT DISTINCT Product, Level1Desc
  FROM hub_prd.g_bpc_finance.v_reltioproduct_hier
  WHERE Level1Desc IN (
    'Alaris System','Hazardous Drug Solutions','IV Solutions',
    'Gravity and Syringe Delivery','IV Access','CME',
    'Alaris Plus','IPD OEM','BD Nexus'
  )
  AND Product IS NOT NULL
),
bounds AS (
  SELECT
    -- Fiscal year starts in October (month 10) → shift by -9 to anchor on the fiscal year
    add_months(trunc(add_months(current_date, -9), 'YYYY'), 9)  AS fy_start,      -- Oct 1 (current FY)
    add_months(trunc(add_months(current_date, -9), 'YYYY'), 21) AS fy_end_excl,   -- Oct 1 (next FY, exclusive)
    add_months(trunc(add_months(current_date, -9), 'YYYY'), -3) AS prev_fy_start  -- Oct 1 (previous FY)
),
bounds_str AS (
  SELECT
    date_format(prev_fy_start, 'yyyyMM')                      AS start_yyyymm,
    date_format(add_months(fy_end_excl, -1), 'yyyyMM')        AS end_yyyymm_incl
  FROM bounds
)
SELECT f.*
FROM hub_prd.s_shared.scm_histdmd f
JOIN material_set m
  ON f.Product = m.Product
CROSS JOIN bounds_str b
WHERE CAST(f.CalMonth AS STRING) BETWEEN b.start_yyyymm AND b.end_yyyymm_incl
AND f.SnapshotDate = (
  SELECT MAX(SnapshotDate)
  FROM hub_prd.s_shared.scm_histdmd
)